# SaRoHead - Reptile meta-learning

Needs an A100/L4. The auxiliary datasets must be on Drive as `MyDrive/sarohead/unsupervised_TTL/{click,saroco,scitechbait}/train.csv`. The Romanian BERT checkpoints of seed 42 are kept for the SHAP analysis.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/sarohead")
REPO = Path("/content/SaRoHead")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/GRAI-UNSTPB/SaRoHead.git", str(REPO)], check=True)

RESULTS = DRIVE / "results" / "predictions"
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["SAROHEAD_RESULTS"] = str(RESULTS)
sys.path.insert(0, str(REPO))
from analysis_scripts.prediction_io import prediction_filename

SEEDS = [13, 42, 123, 2024, 7]
CATEGORIES = ["social", "politic", "sport"]


def run(cmd, cwd):
    print(">>", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd), check=True)


def done(method, model, variant, category, seed):
    return (RESULTS / method / prediction_filename(method, model, variant, category, seed, "random", "test")).exists()


In [ ]:
!pip install -q lightning torchmetrics transformers pandas scikit-learn

MODELS = ["bert", "distilled-bert", "xlm-roberta"]
CKPT_OUT = DRIVE / "results" / "checkpoints" / "reptile"

for task in ["click", "saroco", "scitechbait"]:
    dst = REPO / "unsupervised_TTL" / task / "train.csv"
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE / "unsupervised_TTL" / task / "train.csv", dst)


In [ ]:
for seed in SEEDS:
    for model in MODELS:
        cats = [c for c in CATEGORIES if not done("reptile", model, "reptile", c, seed)]
        if not cats:
            continue
        run([sys.executable, "train_reptile.py", "--model_type", model, "--train_style", "separate", "--K", 7,
             "--seed", seed, "--categories", *cats], REPO / "reptile")
        for c in cats:
            ckpt = REPO / "reptile" / "backup" / f"separate_meta_learning_{model}_{c}_random_seed{seed}.pt"
            if model == "bert" and seed == 42:
                CKPT_OUT.mkdir(parents=True, exist_ok=True)
                shutil.copy2(ckpt, CKPT_OUT / ckpt.name)
            ckpt.unlink(missing_ok=True)


In [ ]:
files = sorted(RESULTS.rglob("*.csv"))
print(len(files), "prediction files in", RESULTS)
